# Logistic Regression with Smile

Smile is the most concise of the four libraries in this post. Its logistic
regression takes a plain `double[][]` of features and an `int[]` of labels and
hands back a model — the API a scikit-learn refugee expects.

All the data work — loading `results.csv`, canonicalizing team names, building
Elo and recent-form features, splitting chronologically — lives in the shared
`.java` files we `%load` below. This notebook only does the Smile-specific part:
**adapt the feature table to Smile's structure, fit, and read the result.**

In [1]:
// Keep Smile's SLF4J logging at warn so the optimizer's INFO chatter stays out
// of our output. Must be set before the logger initializes, so it goes first.
System.setProperty("org.slf4j.simpleLogger.defaultLogLevel", "warn");

In [2]:
%%loadFromPOM
<dependency>
    <groupId>com.fasterxml.jackson.dataformat</groupId>
    <artifactId>jackson-dataformat-csv</artifactId>
    <version>2.17.2</version>
</dependency>
<dependency>
    <groupId>com.github.haifengl</groupId>
    <artifactId>smile-core</artifactId>
    <version>3.1.1</version>
</dependency>
<!-- An SLF4J binding so Smile's logging facade has a provider. Without this you
     get a harmless "No SLF4J providers were found" warning on first use. We set
     the level to warn (see the cell above) so routine optimizer logs stay quiet. -->
<dependency>
    <groupId>org.slf4j</groupId>
    <artifactId>slf4j-simple</artifactId>
    <version>2.0.12</version>
</dependency>

## The shared pipeline

Loading, name canonicalization, feature engineering, the chronological split,
the fair metrics, and the fixture printout are identical across every library in
this post, so they live once in `shared/`. See the post for what each one does.

In [3]:
%load shared/Match.java
%load shared/DataLoader.java
%load shared/FormerName.java
%load shared/TeamNames.java
%load shared/EloRating.java
%load shared/RecentForm.java
%load shared/FeatureRow.java
%load shared/FeatureEngineering.java
%load shared/TrainTestSplit.java
%load shared/Metrics.java
%load shared/Predictions.java

In [4]:
var all = DataLoader.loadAll("/home/jovyan/data/results.csv");
var names = TeamNames.load("/home/jovyan/data/former_names.csv");
var fe = FeatureEngineering.build(all, names);
var split = TrainTestSplit.chronological(fe.played(), 0.8);

System.out.println("train: " + split.train().size() + "   test: " + split.test().size());
System.out.println("upcoming fixtures to predict: " + fe.upcoming().size());

train: 39546   test: 9887
upcoming fixtures to predict: 44


## The Smile adapter

Here is the only library-specific data step: turn `List<FeatureRow>` into the
primitive arrays Smile wants. Our shared `TrainTestSplit.toX/toY` already do
exactly this, so the adapter is a single line each.

In [5]:
double[][] trainX = TrainTestSplit.toX(split.train());
int[]      trainY = TrainTestSplit.toY(split.train());
double[][] testX  = TrainTestSplit.toX(split.test());
int[]      testY  = TrainTestSplit.toY(split.test());

System.out.println("feature columns: " + java.util.Arrays.toString(FeatureRow.featureNames()));

feature columns: [eloDiff, homeWinRate, awayWinRate, homeGoalDiff, awayGoalDiff, neutral]


## Train

This is the whole training step. One line.

In [6]:
import smile.classification.LogisticRegression;

var model = LogisticRegression.fit(trainX, trainY);

## Evaluate

Smile's `predict(sample, posteriori)` fills a 2-element array with the class
probabilities; index `1` is P(home win). We collect those probabilities and hand
them to our shared `Metrics` so the score is computed the same way for every
library.

In [7]:
double[] testProbs = new double[testX.length];
for (int i = 0; i < testX.length; i++) {
    double[] posteriori = new double[2];
    model.predict(testX[i], posteriori);
    testProbs[i] = posteriori[1]; // P(home win)
}

var metrics = Metrics.from(testProbs, testY);
System.out.println("Smile logistic regression");
System.out.println(metrics);

Smile logistic regression
n=9887  accuracy=0.711  precision=0.680  recall=0.742  f1=0.710  logLoss=0.5561  brier=0.1889


## Read the coefficients

This is supposedly why you reach for logistic regression in the first place.
Smile's binary model exposes its learned coefficients; the last entry is the
intercept. Each coefficient, exponentiated, is an **odds ratio**: how the odds of
a home win move when that feature increases by one unit, holding the rest fixed.

Read the numbers below carefully before you trust them — we come back to them.

In [8]:
var binomial = (LogisticRegression.Binomial) model;
double[] coef = binomial.coefficients();
String[] featureNames = FeatureRow.featureNames();

System.out.printf("%-14s %12s %12s%n", "feature", "coefficient", "odds ratio");
for (int i = 0; i < featureNames.length; i++) {
    System.out.printf("%-14s %12.5f %12.3f%n",
        featureNames[i], coef[i], Math.exp(coef[i]));
}
System.out.printf("%-14s %12.5f%n", "(intercept)", coef[coef.length - 1]);
System.out.flush();

feature         coefficient   odds ratio
eloDiff             0.00591        1.006
homeWinRate        -0.94417        0.389
awayWinRate         0.87441        2.397
homeGoalDiff        0.18311        1.201
awayGoalDiff       -0.23243        0.793
neutral            -0.38130        0.683
(intercept)         0.05487


### Wait — those coefficients are lying to you

Look at the form features. `homeWinRate` has a **negative** coefficient (odds
ratio ~0.39) and `awayWinRate` a **positive** one (~2.40). Taken literally, the
model is claiming that *the better the home team's recent form, the less likely
they are to win* — and the opposite for the away team. That is nonsense, and it's
worth sitting with rather than quietly moving past.

The accuracy didn't lie: ~0.71 with a solid log-loss. The model *predicts* fine.
But the **coefficients** are not safe to interpret, for two compounding reasons:

- **Scale.** `eloDiff` lives in the hundreds; the rates live in `[0, 1]`. With no
  standardization, the optimizer hands `eloDiff` a tiny per-unit coefficient
  (0.006 — but multiplied by a 300-point gap, that's the real engine of the
  model) while the small-range features absorb leftover, unstable weight.
- **Collinearity.** `homeWinRate`, `awayWinRate`, and `eloDiff` all encode
  "who's better." When features move together, logistic regression can split the
  credit between them almost arbitrarily, including with flipped signs that still
  fit the training data.

This is the whole reason interpretability is a feature and not a given. A model
can be **accurate and uninterpretable at the same time**. Reading coefficients
off an unscaled, collinear logistic regression and reporting them as "insights"
is how a perfectly good prediction becomes a wrong explanation in a meeting.

We're deliberately leaving the features raw so you can see this happen. The fixes
are standard — standardize the inputs, drop or combine the collinear features,
or add regularization — and we'll keep them in mind when we build the production
churn model in the next post. For now: trust this model's *predictions*, not its
*story*.

## Predict the 2026 World Cup group stage

The payoff. These 44 matches have no result in the data — they're the 2026 FIFA
World Cup group stage (June 19–27), which hadn't been played when the dataset
was cut. We compute their features from history and let the model call them.

In [9]:
double[][] upcomingX = TrainTestSplit.toX(fe.upcoming());
double[] upcomingProbs = new double[upcomingX.length];
for (int i = 0; i < upcomingX.length; i++) {
    double[] posteriori = new double[2];
    model.predict(upcomingX[i], posteriori);
    upcomingProbs[i] = posteriori[1];
}

Predictions.print(fe.upcoming(), upcomingProbs);

date         home                   away                    P(home)   call
2026-06-19   Scotland               Morocco                   21.0%   no home win
2026-06-19   Brazil                 Haiti                     82.6%   Brazil win
2026-06-19   United States          Australia                 50.1%   United States win
2026-06-19   Turkey                 Paraguay                  48.9%   no home win
2026-06-20   Germany                Ivory Coast               64.1%   Germany win
2026-06-20   Ecuador                Curaçao                   88.5%   Ecuador win
2026-06-20   Netherlands            Sweden                    61.5%   Netherlands win
2026-06-20   Tunisia                Japan                     13.0%   no home win
2026-06-21   Belgium                Iran                      47.4%   no home win
2026-06-21   New Zealand            Egypt                     23.7%   no home win
2026-06-21   Spain                  Saudi Arabia              84.3%   Spain win
2026-06-21   Uru